# NB2 - Reflexion: Reflection is the Gradient

**Workshop: Self-Evolving Agents by Optimizing the Harness (no GPU)**

In NB0 you built the agent; in NB1 you measured it (the reward **V**). The agent
has one glaring weakness: it has **no memory (S)**. Every question starts from
scratch, so it makes the same class of mistake forever.

Now we fix that - with **zero weight updates**. The mechanism is **Reflexion**
(Shinn et al.): the agent attempts tasks, compares its answers to feedback,
writes **lessons in natural language**, stores them, and consults them on future
questions. In harness terms (H = E, T, **C**, **S**, L, V) we are filling in:
- **S (state store)** - lessons that persist across tasks, and
- **C (context)** - we inject the relevant lessons back into the prompt.

This is the seed of the **skill library** we formalize in NB3-NB5: raw experience
-> distilled lessons -> reused on new problems.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
from workshop_utils import (
    build_db, load_tasks, run_sql, score_sql, evaluate,
    llm, METER, SCHEMA_TEXT, extract_sql, baseline_prompt, make_agent,
    preflight, flush,
)
preflight("WANDB_API_KEY")  # OPENAI + LANGFUSE + W&B keys (see SETUP.md)
build_db()
agent = make_agent()      # the NB0 agent (no memory yet)

# Recap the NB1 baseline number (recompute if NB1 wasn't run this session).
try:
    baseline_acc = json.load(open("../data/baseline_test.json"))["accuracy"]
except FileNotFoundError:
    baseline_acc = evaluate(agent, split="test")["accuracy"]
print("baseline test accuracy:", round(baseline_acc, 3))

## Learn across tasks: distill lessons from the train split

The recipe:
1. Run the agent on each **train** task (where we DO have gold labels).
2. On every failure, distill **one general, reusable lesson** - by comparing the
   wrong SQL to the gold SQL. The lesson must generalize, not memorize the answer.
3. Store the lessons in memory (**S**) and inject them into the prompt (**C**).
4. Measure on **test** as the memory grows - the *learning curve*.

This is Reflexion as **harness evolution**: the weights never change; the agent
gets better because its *state* and *context* got better.

In [ ]:
LESSON_SYS = (
    "You distill a single GENERAL, reusable lesson from a SQL mistake. "
    "The lesson must help on FUTURE, different questions - never mention the "
    "specific question or specific values. Keep it to at most two sentences."
)

def distill_lesson(question, wrong_sql, gold_sql):
    msgs = [
        {"role": "system", "content": LESSON_SYS},
        {"role": "user", "content":
            "Schema:\n" + SCHEMA_TEXT +
            "\nQuestion: " + question +
            "\n\nMy incorrect SQL:\n" + wrong_sql +
            "\n\nThe correct SQL:\n" + gold_sql +
            "\n\nWrite ONE general lesson to avoid this class of mistake."},
    ]
    return llm(msgs).strip()

train = [t for t in load_tasks() if t["split"] == "train"]
lessons = []
METER.reset()
for t in train:
    sql = agent(t["question"])
    if not score_sql(sql, t["gold"]):
        lessons.append(distill_lesson(t["question"], sql, t["gold"]))
print("collected", len(lessons), "lessons from", len(train), "train tasks\n")
for l in lessons:
    print("-", l)
print("\n", METER)

### Inject the lessons and watch the test number move

`make_agent(extra=...)` reuses the *same* NB0 harness, but now we prepend the
learned lessons to the context (**C**). We grow the memory in steps and re-measure
**test** each time - the "learning curve" of a self-evolving agent.

We treat this exactly like a model training run and log it to **Weights &
Biases**: each added lesson is a "step", and the held-out test accuracy is the
metric. This is the same pattern you'd use to track a real fine-tune - here the
"training" is happening in text space.

In [ ]:
import wandb

def memory_block(lessons):
    if not lessons:
        return ""
    return ("Lessons learned from past mistakes (apply when relevant):\n" +
            "\n".join("- " + l for l in lessons))

def make_memory_agent(lessons):
    # Same harness as NB0, with memory (S) injected into the context (C).
    return make_agent(extra=memory_block(lessons))

# Open a W&B run; lessons_in_memory is our x-axis (the "training step").
run = wandb.init(project="rl-agents-workshop", name="nb2-reflexion",
                 config={"model": "gpt-4o-mini", "n_train": len(train),
                         "n_lessons": len(lessons)})
wandb.define_metric("lessons_in_memory")
wandb.define_metric("test_accuracy", step_metric="lessons_in_memory")

checkpoints = sorted(set([0, len(lessons) // 3, 2 * len(lessons) // 3, len(lessons)]))
curve = []
METER.reset()
for k in checkpoints:
    acc = evaluate(make_memory_agent(lessons[:k]), split="test")["accuracy"]
    curve.append((k, acc))
    wandb.log({"lessons_in_memory": k, "test_accuracy": acc})
    print("lessons in memory = %2d   test accuracy = %.3f" % (k, acc))

wandb.summary["baseline_acc"] = baseline_acc
wandb.summary["final_acc"] = curve[-1][1]
wandb.summary["est_cost_usd"] = METER.cost()
print("\n", METER)
print("W&B run:", run.url)

In [ ]:
import matplotlib.pyplot as plt
xs = [c[0] for c in curve]
ys = [c[1] for c in curve]
plt.figure(figsize=(6, 4))
plt.plot(xs, ys, marker="o", label="reflection memory")
plt.axhline(baseline_acc, ls="--", color="gray", label="NB0 agent baseline")
plt.xlabel("# lessons in memory (state S)")
plt.ylabel("test accuracy")
plt.title("Reflexion: accuracy rises as the harness learns (weights frozen)")
plt.ylim(0, 1)
plt.legend()
plt.show()

## Failure mode - memory pollution

Self-evolution is not free of risk. A bad or over-general lesson can *lower*
accuracy. The Microsoft/Fudan study (NB3) found **25% of skill pairings actually
degrade performance**. Here we inject a plausible-but-harmful lesson and watch
the number drop - motivating the **validation gates** of SkillOpt (NB4).

In [ ]:
bad_lessons = ["To be safe, always add LIMIT 1 to every query.",
               "Always wrap every aggregate in a subquery."]
polluted = evaluate(make_memory_agent(lessons + bad_lessons * 2), split="test")["accuracy"]
print("clean memory:", round(curve[-1][1], 3), " ->  polluted memory:", round(polluted, 3))

wandb.summary["polluted_acc"] = polluted      # record the failure mode on the run
wandb.finish()                                # close the W&B run
flush()                                       # ship traces to Langfuse

## Takeaways

- **Reflection is the gradient.** The agent improved on held-out test with zero
  weight updates - only the harness (state **S** + context **C**) changed.
- Memory turns a one-shot agent into one that **learns from its own mistakes**.
- Self-evolution can go **backwards** (memory pollution). You need a gate.
- We tracked the whole thing in **Weights & Biases** like any training run - the
  learning curve, the baseline, the cost, and the pollution result all live on one
  run you can share and compare. NB4 reuses this to track SkillOpt.

### The gap this leaves (-> NB3, NB4)
These lessons are **unstructured free text**, **unvalidated**, and we dump *all*
of them into every prompt. The skill-optimization papers fix exactly this:
- **NB3 (skill lifecycle):** turn raw experience into *structured* skills, retrieve
  only the relevant ones, and learn the 25%-degrade trap + the meta-skill rubric.
- **NB4 (SkillOpt):** treat the skill document as a *trainable parameter* with a
  learning rate, **validation gate**, and momentum - so memory can't pollute.

### Exercise
1. Make `distill_lesson` produce lessons WITHOUT seeing the gold SQL (only the
   execution result). Are the lessons still useful? Why / why not?
2. Instead of injecting *all* lessons, retrieve only the 2-3 most relevant to each
   question (e.g. keyword overlap). Does test accuracy improve? This previews the
   retrieval step of NB3.